In [ ]:
import sys
import os
from pathlib import Path

import torch
import torch.nn as nn

sys.path.append(os.path.abspath("../src"))

from models import SimpleCNN, ResNet18, DenseNet121, EfficientNetB0, MobileNetV2, ShuffleNetV2, SqueezeNet

from data import (
    prepare_full_dataframe, 
    prepare_data, 
    sample_image_path,
    get_transforms
)

from train_eval import (
    setup_training,
    train_model,
    evaluate,
    predict_single_image
)

from utils import (
    get_device,
    plot_training_history,
    plot_confusion_matrix_figure,
    get_model_path
)
import config

# Data Preprocessing

In [ ]:
dataset_path = "/Volumes/Secretary/Datasets/NIH_Chest_X-Rays"
print("Dataset location:", dataset_path)

metadata_file = os.path.join(dataset_path, "Data_Entry_2017.csv")
df = prepare_full_dataframe(metadata_file, dataset_path)

print("Total images:", len(df))
print("Unique patients:", df["Patient ID"].nunique())
df["Finding Labels"].value_counts().head()

In [ ]:
print(df["split"].value_counts())

In [ ]:
train_loader, val_loader, test_loader = prepare_data(df)
device = get_device()

In [ ]:
model_registry = {
    "CNN": SimpleCNN,
    "ResNet18": lambda: ResNet18(num_classes=2, in_channels=1),
    "DenseNet121": lambda: DenseNet121(num_classes=2, in_channels=1),
    "EfficientNet-B0": lambda: EfficientNetB0(num_classes=2, in_channels=1),
    "MobileNetV2": lambda: MobileNetV2(num_classes=2, in_channels=1),
    "ShuffleNetV2": lambda: ShuffleNetV2(num_classes=2, in_channels=1),
    "SqueezeNet": lambda: SqueezeNet(num_classes=2, in_channels=1)
}

model = SqueezeNet(num_classes=2, in_channels=1).to(device)
criterion, optimizer = setup_training(model)

In [ ]:
results = []

for model_name, model_builder in model_registry.items():
    print("=" * 80)
    print(f"Starting experiment: {model_name}")
    print("=" * 80)

    # Instantiate model
    model = model_builder().to(device)

    # Loss and optimizer
    criterion, optimizer = setup_training(model)

    # Train (train_model uses config.NUM_EPOCHS and saves best model automatically)
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    # Load best weights before testing
    save_path = config.MODEL_SAVE_DIR / f"best_{model_name.lower().replace('-', '_')}.pt"
    model.load_state_dict(torch.load(save_path, map_location=device))

    # Test (evaluate expects 'loader' not 'dataloader')
    test_loss, test_acc, test_precision, test_recall, test_f1, _, _ = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device
    )

    # Store results
    row = {
        "model": model_name,
        "epochs": config.NUM_EPOCHS,
        "batch_size": config.BATCH_SIZE,
        "image_size": config.IMAGE_SIZE,
        "test_loss": test_loss,
        "accuracy": test_acc,
        "precision": test_precision,
        "recall": test_recall,
        "f1": test_f1,
    }

    results.append(row)
    print(f"Finished {model_name}")
    print(row)
    print()

    # Resource cleanup
    del model
    import gc
    gc.collect()

## Model Comparison Plots
Visualize the test results for all models using bar plots for accuracy, precision, recall, and F1 score.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Metrics to plot
metrics = ["accuracy", "precision", "recall", "f1"]

plt.figure(figsize=(12, 6))
results_melted = results_df.melt(id_vars="model", value_vars=metrics, var_name="Metric", value_name="Score")
sns.barplot(data=results_melted, x="model", y="Score", hue="Metric")
plt.title("Model Comparison on Test Set")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()